# Module 1.5 — Practical Python Patterns for Data Work

## Data Science Bootcamp | Interactive Code-Along

> **Today’s outcome:** Write clearer and more reusable Python using sets, constants, documentation, type hints, comprehensions, generators, argument unpacking, and command-line flags.

**Suggested duration:** 3–4 hours. Treat command-line parsing and generators as stretch topics if time is limited.

**Scenario:** A fictional clinic exports simple appointment-administration records. These examples are for learning Python, not for storing real patient information or making clinical decisions.

### Learning objectives
- Remove duplicate values with a set while recognizing that sets do not preserve meaningful row order.
- Name a constant, write a docstring, and add useful type hints.
- Explain why passing a parameter is normally preferable to changing a global variable.
- Unpack a list or dictionary into function arguments.
- Refactor a straightforward loop into a list or dictionary comprehension.
- Use `yield` to produce a sequence one item at a time.
- Read basic CLI flags with `argparse` in a script.

### How to work
Predict the result, run each cell with **Shift + Enter**, then change one input. Run cells in order after restarting the kernel. Empty practice cells are intentionally left for you to complete.

## 1. Sets: find unique values

An appointment extract may contain the same department many times. `set(values)` keeps unique values. A set is useful for a quick uniqueness check or membership test, but do not use it to silently delete duplicate *records*: repeated records may need investigation. Sort a set when you want predictable display order.

**Say live:** ‘Before counting records, I want to know which department labels appear at all. That is a different question from whether two full rows are duplicates.’

In [ ]:
departments = ['Reception', 'General Practice', 'Reception', 'Physiotherapy']
unique_departments = set(departments)
print('Number of records:', len(departments))
print('Number of unique departments:', len(unique_departments))
print('Department labels:', sorted(unique_departments))
print('Is Reception present?', 'Reception' in unique_departments)

### Pause and predict
If the list contains four entries, why does the set contain only three? Would converting a list of appointment IDs to a set tell us which full records are safe to delete?

**EDA connection:** Comparing total rows with distinct IDs helps reveal possible duplicate keys. The difference is a signal to inspect—not automatic permission to drop records.

In [ ]:
appointment_ids = ['APT-1001', 'APT-1002', 'APT-1001', 'APT-1003']
print('Rows:', len(appointment_ids))
print('Distinct IDs:', len(set(appointment_ids)))
print('Possible repeated IDs:', len(appointment_ids) - len(set(appointment_ids)))

## 2. Constants, scope, docstrings, and type hints

A name such as `MAX_DAILY_APPOINTMENTS` signals a value we intend not to change. Python does not enforce this convention at runtime. A docstring states a function’s purpose, and type hints describe the kinds of inputs and results we expect. Type hints are not runtime validation.

Avoid `global` for routine data-processing functions. A function that receives a threshold as a parameter is easier to test and reuse than a function that secretly depends on changing outside state.

In [ ]:
MAX_DAILY_APPOINTMENTS = 20  # Naming convention, not a Python-enforced constant.

def has_capacity(booked_count: int, maximum: int = MAX_DAILY_APPOINTMENTS) -> bool:
    """Return True if another appointment can be booked."""
    return booked_count < maximum

print(has_capacity(18))
print(has_capacity(20))
print(has_capacity(4, maximum=5))
print(has_capacity.__doc__)

### Why not modify a global?
The next cell demonstrates `global`, but it is an **illustration, not the recommended solution**. Run it twice: the answer changes because the function changes shared state. Compare it with a function that takes an explicit input and returns a new value.

**Say live:** ‘If I cannot tell what a function needs from its arguments, reproducing my analysis becomes harder.’

In [ ]:
appointments_booked = 0

def book_with_global() -> None:
    global appointments_booked
    appointments_booked += 1

book_with_global()
book_with_global()
print('Shared count:', appointments_booked)

def next_booking_count(current_count: int) -> int:
    """Return a new count without modifying outside state."""
    return current_count + 1

print('Explicit result:', next_booking_count(0))
print('Explicit result again:', next_booking_count(0))

### Checkpoint
Write `minutes_remaining(booked_minutes: int, available_minutes: int = 480) -> int`. Return the remaining minutes. Write a one-sentence docstring. Test with 120 booked minutes and again with a custom daily limit.

**Stretch:** A static type checker such as `mypy` can flag some mismatches before execution, but it does not replace input validation. From a terminal, after installing it, you can run `python -m mypy your_script.py`.

In [ ]:
# Your solution here.
# def minutes_remaining(booked_minutes: int, available_minutes: int = 480) -> int:
#     ...

## 3. Argument unpacking

Use `*` to unpack a sequence into positional arguments and `**` to unpack a dictionary into named arguments. The number of values and dictionary keys must match the function signature. This is useful when validated fields from a record need to be passed to a calculation.

`*args` and `**kwargs` **inside a function definition** collect an arbitrary number of positional or keyword arguments. Prefer explicit parameter names when the expected inputs are known.

In [ ]:
def appointment_label(patient_name: str, department: str) -> str:
    return f'{patient_name} — {department}'

parts = ['Alex Morgan', 'General Practice']
record = {'patient_name': 'Sam Lee', 'department': 'Physiotherapy'}

print(appointment_label(*parts))       # Sequence -> positional arguments.
print(appointment_label(**record))     # Mapping -> named arguments.

In [ ]:
def describe_visit(patient_name: str, *notes: str, **details: str) -> str:
    """Create a short administrative description from optional details."""
    note_text = ', '.join(notes) if notes else 'no notes'
    detail_text = ', '.join(f'{key}={value}' for key, value in details.items())
    return f'{patient_name}: {note_text}; {detail_text or "no extra details"}'

print(describe_visit('Alex Morgan', 'arrive early', department='Reception', status='Booked'))
print(describe_visit('Sam Lee'))

## Code-Along 1.5.1 — From loops to comprehensions

### Goal
Filter simple records with an ordinary loop, then express the **same** transformation as a list comprehension. Next, build a lookup dictionary with a dictionary comprehension.

### Instructor transcript
Say: ‘The loop is already correct and readable. We are not replacing it because shorter always means better. A comprehension is useful when one simple expression creates each output value.’

Each record is fictional, contains only basic appointment metadata, and has a unique ID.

In [ ]:
appointments = [
    {'id': 'APT-1001', 'department': 'General Practice', 'status': 'Booked'},
    {'id': 'APT-1002', 'department': 'Physiotherapy', 'status': 'Completed'},
    {'id': 'APT-1003', 'department': 'General Practice', 'status': 'Booked'},
    {'id': 'APT-1004', 'department': 'Reception', 'status': 'Cancelled'},
]

booked_ids_loop = []
for appointment in appointments:
    if appointment['status'] == 'Booked':
        booked_ids_loop.append(appointment['id'])

booked_ids_comp = [
    appointment['id']
    for appointment in appointments
    if appointment['status'] == 'Booked'
]

print('Loop:', booked_ids_loop)
print('Comprehension:', booked_ids_comp)
assert booked_ids_loop == booked_ids_comp

In [ ]:
# A dictionary comprehension builds an ID -> department lookup.
department_by_id = {
    appointment['id']: appointment['department']
    for appointment in appointments
}

print(department_by_id)
print('Department for APT-1002:', department_by_id['APT-1002'])

# Real data may contain duplicate IDs. A dictionary would keep the last value,
# so check uniqueness before using IDs as dictionary keys.
ids = [appointment['id'] for appointment in appointments]
assert len(ids) == len(set(ids)), 'Appointment IDs must be unique.'

### Practice
Create a list of IDs for `Completed` appointments. Then make a dictionary that maps each appointment ID to its status. First write a loop; only then refactor it into a comprehension.

**EDA connection:** Filtering records and creating lookup mappings are building blocks for segmentation and data preparation. In pandas, you will do similar work with Boolean masks and mapped columns.

In [ ]:
# Your code here.
# completed_ids = ...
# status_by_id = ...

## Code-Along 1.5.2 — Generators and `yield`

A list is built and stored in memory. A **generator** produces its values as you iterate over it. `yield` pauses the function and returns one value at a time; a later iteration resumes it.

This matters when data is too large to load all at once. A generator is also helpful when records arrive over time. **Note:** A generator is generally exhausted after one full pass; create a new one to iterate again.

### Instructor transcript
Say: ‘We will demonstrate with just five values so the behavior is visible. The same pattern can later process millions of rows without creating a list of all outputs first.’

In [ ]:
def appointment_numbers(count: int):
    """Yield simple appointment sequence numbers, one at a time."""
    for number in range(1, count + 1):
        yield number

numbers = appointment_numbers(5)
print('Generator object:', numbers)
print('First value:', next(numbers))
print('Remaining values:', list(numbers))
print('After consumption:', list(numbers))
print('New generator:', list(appointment_numbers(5)))

In [ ]:
def booked_records(records):
    """Yield booked appointment records from an iterable."""
    for record in records:
        if record['status'] == 'Booked':
            yield record

for record in booked_records(appointments):
    print(record['id'], record['department'])

### Practice
Write a generator called `high_waiting_times(values, threshold)` that yields values strictly greater than the threshold. Try it with `[5, 20, 12, 35]` and a threshold of `15`. Explain what happens if you consume the same generator twice.

In [ ]:
# Your code here.
# def high_waiting_times(values, threshold):
#     ...

## 6. Command-line flags with `argparse`

When a script needs optional settings, named flags are clearer than remembering the position of each item in `sys.argv`. `argparse` is in Python’s standard library; it generates help text and parses arguments.

A notebook kernel already has its own launch arguments. To keep this cell reproducible, we pass a small **simulated list** into `parse_args()` instead of reading the kernel’s real arguments.

In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Summarize fictional appointments.')
parser.add_argument('--department', default='All', help='Department to report on.')
parser.add_argument('--limit', type=int, default=10, help='Maximum records to show.')

options = parser.parse_args(['--department', 'Physiotherapy', '--limit', '5'])
print('Department:', options.department)
print('Limit:', options.limit)

defaults = parser.parse_args([])
print('Default values:', defaults.department, defaults.limit)

### Script version for a terminal
Save the following in `appointment_report.py`. Do **not** paste its `parse_args()` call into an ordinary notebook cell without providing an explicit argument list.

```python
import argparse

def main() -> None:
    parser = argparse.ArgumentParser(description='Summarize fictional appointments.')
    parser.add_argument('--department', default='All')
    parser.add_argument('--limit', type=int, default=10)
    options = parser.parse_args()
    print(f'Department: {options.department}; record limit: {options.limit}')

if __name__ == '__main__':
    main()
```

Run from a terminal:

```bash
python appointment_report.py --department Physiotherapy --limit 5
python appointment_report.py --help
```

**Stretch:** What happens if the user types a word instead of a number for `--limit`? How could you reject negative limits with a separate validation rule?

## Integrated practice — A tiny daily report

Use the fictional `appointments` list from the comprehension code-along. Write small, named functions to:

1. Return the number of unique departments.
2. Return booked appointment IDs.
3. Return a dictionary mapping each ID to its status.
4. Return a generator that yields only records for a selected department.
5. Add docstrings and type hints to at least two functions.

Before turning records into a dictionary keyed by ID, check that the IDs are unique. Print a short report using **fictional data only**.

**DS connection:** The same combination of filtering, quality checks, reusable transformations, and documented outputs appears in EDA preparation and ML feature pipelines.

In [ ]:
# Your integrated solution here.
# def count_unique_departments(records):
#     ...
# def get_booked_ids(records):
#     ...
# def records_for_department(records, department):
#     yield ...

## Debugging and teaching notes

- A set identifies unique **values**; it is not a substitute for investigating duplicate rows.
- Type hints describe intentions; they do not automatically convert or validate input.
- `global` makes shared state mutable; prefer explicit inputs and returned results for ordinary analysis functions.
- A comprehension is not automatically better than a clear loop. Use loops when logic needs several steps or error handling.
- A generator can be consumed once. Recreate it when you need another pass.
- Duplicate dictionary keys overwrite earlier values. Check ID uniqueness when the distinction matters.
- In Jupyter, call `parse_args([...])` with explicit values; a script can call `parse_args()` normally.

### Exit ticket
1. When would a set help you during early data exploration?
2. Why is a passed-in threshold easier to test than a modified global?
3. What is the difference between `*values` and `**record` at a function call?
4. Rewrite `for item in items: if condition: result.append(item)` as a list comprehension.
5. Why might a generator be preferable to a list for a large file?
6. Why is `argparse` helpful for a script with several optional settings?

**Next:** NumPy arrays, vectorized calculations, and the transition from core Python collections to numerical data analysis.